# Benchmark: Spark CPU vs GPU (RAPIDS Accelerator) en Google Colab T4

**Instituto Tecnológico Autónomo de México**  
**Ingeniería en Ciencia de Datos — Arquitectura de Grandes Volúmenes de Datos**  
**Prof. Wilmer Efrén Pereira González**  
**Fecha:** 2026-05-10

---

## Objetivo

Comparar el desempeño de Apache Spark ejecutando operaciones batch SQL (groupBy, aggregations, joins, correlaciones) sobre datos de criptoactivos en:

- **CPU puro** (2 vCPUs de Colab)
- **GPU T4** con RAPIDS Accelerator for Apache Spark (25.02.0)

Los resultados complementan la Arquitectura 1 (CPU local i7-12700H) documentada en el reporte principal.

### ¿Qué acelera RAPIDS?

| Operación | ¿GPU? | Nota |
|-----------|--------|------|
| `groupBy + agg` | ✓ | 2-5× speedup típico |
| `percentile_approx` | ✓ | T-Digest en GPU |
| `broadcast join` | ✓ | Vectorizado |
| `F.corr` (correlación) | ✓ | cuDF nativo |
| Streaming stateful (watermark, state store) | ✗ | Sin soporte RAPIDS |
| MLlib RandomForest | ✗ | Sin backend cuML |

---
## 1. Verificación de GPU

Confirmar que el runtime tiene una T4 asignada. Si falla, ir a **Runtime → Change runtime type → T4 GPU**.

In [1]:
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "NO GPU DETECTED.\n"
        "Ve a Runtime > Change runtime type > T4 GPU y vuelve a ejecutar."
    )
print(result.stdout)

assert 'T4' in result.stdout, "Se esperaba GPU T4. Verifica el runtime type."
print("\n[OK] NVIDIA T4 detectada (16 GB VRAM)")

Sun May 10 19:32:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             28W /   70W |     121MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## 2. Instalación de dependencias

- PySpark 3.5.2 (alineado con RAPIDS 25.02)
- RAPIDS Accelerator JAR desde Maven Central (~200 MB, bundlea cuDF)

In [2]:
%%time
import os

!pip install pyspark==3.5.2 -q

# RAPIDS 24.10.1 (version estable documentada) - si 25.02.0 causa hang, usar esta
RAPIDS_VERSION = "24.10.1"
RAPIDS_JAR = f"/content/rapids-4-spark_2.12-{RAPIDS_VERSION}.jar"

if not os.path.exists(RAPIDS_JAR):
    url = f"https://repo1.maven.org/maven2/com/nvidia/rapids-4-spark_2.12/{RAPIDS_VERSION}/rapids-4-spark_2.12-{RAPIDS_VERSION}.jar"
    print(f"Descargando RAPIDS {RAPIDS_VERSION}...")
    !wget -q {url} -O {RAPIDS_JAR}

jar_size = os.path.getsize(RAPIDS_JAR) / (1024**2)
assert jar_size > 50, f"JAR demasiado pequeño ({jar_size:.1f} MB) — descarga fallida"

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f"--driver-class-path {RAPIDS_JAR} "
    f"--jars {RAPIDS_JAR} "
    f"pyspark-shell"
)

print(f"\n[OK] pyspark==3.5.2 + RAPIDS JAR {RAPIDS_VERSION} ({jar_size:.0f} MB)")
print(f"[OK] PYSPARK_SUBMIT_ARGS configurado")


[OK] pyspark==3.5.2 + RAPIDS JAR 24.10.1 (516 MB)
[OK] PYSPARK_SUBMIT_ARGS configurado
CPU times: user 2.47 s, sys: 443 ms, total: 2.92 s
Wall time: 14.3 s


---
## 3. Configuración GPU Discovery

RAPIDS en modo `local[*]` necesita un script que reporte las GPUs disponibles.

In [3]:
import os, stat

DISCOVERY_SCRIPT = "/content/getGpusResources.sh"
with open(DISCOVERY_SCRIPT, "w") as f:
    f.write('#!/bin/bash\necho \'{"name": "gpu", "addresses": ["0"]}\'\n')
os.chmod(DISCOVERY_SCRIPT, os.stat(DISCOVERY_SCRIPT).st_mode | stat.S_IEXEC)

print(f"[OK] GPU discovery script: {DISCOVERY_SCRIPT}")
!cat {DISCOVERY_SCRIPT}

[OK] GPU discovery script: /content/getGpusResources.sh
#!/bin/bash
echo '{"name": "gpu", "addresses": ["0"]}'


---
## 4. Generación de datos sintéticos

Replica la lógica de `gen_synth_parquet.py`: 970,000 filas × 10 símbolos cripto = **9.7 millones de filas**.

Features generadas: VWAP, precio promedio, volatilidad, volumen, trade count, spread proxy, bid/ask, min/max, timestamp.

In [4]:
%%time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROWS_PER_SYMBOL = 970_000
SYNTH_PARQUET = "/content/synth.parquet"

# driver.memory=6g aqui porque la JVM se crea UNA VEZ en el kernel.
# Sesiones posteriores heredan este heap aunque pidan otro valor.
spark_gen = (
    SparkSession.builder
    .appName("gen-synth-colab")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)
spark_gen.sparkContext.setLogLevel("WARN")

total_rows = ROWS_PER_SYMBOL * 10
idx = (F.col("id") % F.lit(10)).cast("int")
id_d = F.col("id").cast("double")

symbol_expr = (
    F.when(idx == 0, "BTCUSDT").when(idx == 1, "ETHUSDT")
    .when(idx == 2, "BNBUSDT").when(idx == 3, "SOLUSDT")
    .when(idx == 4, "XRPUSDT").when(idx == 5, "ADAUSDT")
    .when(idx == 6, "DOGEUSDT").when(idx == 7, "AVAXUSDT")
    .when(idx == 8, "DOTUSDT").otherwise("MATICUSDT")
)
base_price_expr = (
    F.when(idx == 0, 65000.0).when(idx == 1, 3500.0)
    .when(idx == 2, 580.0).when(idx == 3, 170.0)
    .when(idx == 4, 0.65).when(idx == 5, 0.45)
    .when(idx == 6, 0.18).when(idx == 7, 38.0)
    .when(idx == 8, 8.5).otherwise(0.85)
)
noise = F.lit(1.0) + F.sin(id_d * F.lit(0.001)) * F.lit(0.02)

df = (
    spark_gen.range(total_rows)
    .withColumn("symbol", symbol_expr)
    .withColumn("_base", base_price_expr)
    .withColumn("vwap", F.col("_base") * noise)
    .withColumn("avg_price", F.col("_base") * (noise + F.lit(0.001)))
    .withColumn("price_volatility", F.abs(F.sin(id_d)) * F.lit(0.05))
    .withColumn("total_volume", F.lit(1000.0) + F.cos(id_d) * F.lit(500.0))
    .withColumn("trade_count", (F.col("id") % F.lit(200) + F.lit(10)).cast("long"))
    .withColumn("avg_spread_proxy", F.col("_base") * F.lit(0.0005))
    .withColumn("avg_best_bid_price", F.col("vwap") * F.lit(0.9995))
    .withColumn("avg_best_ask_price", F.col("vwap") * F.lit(1.0005))
    .withColumn("min_price", F.col("vwap") * F.lit(0.995))
    .withColumn("max_price", F.col("vwap") * F.lit(1.005))
    .withColumn("window_start", (F.lit(1_715_000_000) + F.col("id") * F.lit(5)).cast("timestamp"))
    .drop("id", "_base")
)

df.write.mode("overwrite").partitionBy("symbol").parquet(SYNTH_PARQUET)
row_count = spark_gen.read.parquet(SYNTH_PARQUET).count()
print(f"\n[OK] {row_count:,} filas generadas -> {SYNTH_PARQUET}")
spark_gen.stop()


[OK] 9,700,000 filas generadas -> /content/synth.parquet
CPU times: user 149 ms, sys: 29.9 ms, total: 179 ms
Wall time: 2min 33s


---
## 5. Benchmark CPU (sin RAPIDS)

Ejecuta las mismas operaciones que `gpu_smoke_test.py` usando solo los 2 vCPUs de Colab:
1. `groupBy("symbol").agg(...)` — 13 agregaciones incluyendo `percentile_approx`
2. Correlación con BTCUSDT via `broadcast join` + `F.corr`

In [5]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark_cpu = (
    SparkSession.builder
    .appName("smoke-test-CPU")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark_cpu.sparkContext.setLogLevel("WARN")

df_cpu = spark_cpu.read.parquet(SYNTH_PARQUET)
n_cpu = df_cpu.count()
print(f"Filas le\u00eddas: {n_cpu:,}")

# === Benchmark ===
t0_cpu = time.time()

# 1) groupBy + 13 agregaciones
stats_cpu = df_cpu.groupBy("symbol").agg(
    F.count("*").alias("total_windows"),
    F.avg("vwap").alias("mean_vwap"),
    F.stddev("vwap").alias("stddev_vwap"),
    F.min("avg_price").alias("historical_min_price"),
    F.max("avg_price").alias("historical_max_price"),
    F.avg("price_volatility").alias("mean_volatility"),
    F.max("price_volatility").alias("peak_volatility"),
    F.avg("total_volume").alias("avg_volume_per_window"),
    F.sum("total_volume").alias("total_historical_volume"),
    F.avg("avg_spread_proxy").alias("mean_spread"),
    F.avg("trade_count").alias("avg_trades_per_window"),
    F.percentile_approx("vwap", 0.50).alias("vwap_p50"),
    F.percentile_approx("price_volatility", 0.95).alias("volatility_p95"),
)
stats_cpu.collect()
t_agg_cpu = time.time() - t0_cpu

# 2) Correlaci\u00f3n con BTC via broadcast join
t1_cpu = time.time()
btc = df_cpu.filter(F.col("symbol") == "BTCUSDT").select(
    F.col("window_start"), F.col("vwap").alias("btc_vwap")
)
corr_cpu = (
    df_cpu.join(F.broadcast(btc), on="window_start", how="inner")
    .groupBy("symbol")
    .agg(F.corr("vwap", "btc_vwap").alias("corr_with_btc"))
    .orderBy(F.col("corr_with_btc").desc())
)
corr_cpu.collect()
t_corr_cpu = time.time() - t1_cpu

t_total_cpu = time.time() - t0_cpu

print(f"\n{'='*50}")
print(f"CPU BENCHMARK (Colab - 2 vCPUs Xeon)")
print(f"{'='*50}")
print(f"  groupBy + agg (13 stats):  {t_agg_cpu:.2f}s")
print(f"  BTC correlation (join):    {t_corr_cpu:.2f}s")
print(f"  TOTAL:                     {t_total_cpu:.2f}s")
print(f"{'='*50}")

cpu_results = {
    "agg_time": t_agg_cpu,
    "corr_time": t_corr_cpu,
    "total_time": t_total_cpu,
}

spark_cpu.stop()

Filas leídas: 9,700,000

CPU BENCHMARK (Colab - 2 vCPUs Xeon)
  groupBy + agg (13 stats):  35.56s
  BTC correlation (join):    10.73s
  TOTAL:                     46.30s


---
## 6. Benchmark GPU (RAPIDS Accelerator 25.02.0)

Mismas operaciones con el plugin `com.nvidia.spark.SQLPlugin` habilitado.  
Se incluye un **warmup pass** antes de medir (la primera operación GPU tiene overhead de inicialización CUDA/JIT).

In [6]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import SparkContext

# Asegurar que no hay SparkContext activo previo.
if SparkContext._active_spark_context is not None:
    SparkContext._active_spark_context.stop()

spark_gpu = (
    SparkSession.builder
    .appName("smoke-test-GPU")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.extraClassPath", RAPIDS_JAR)
    .config("spark.executor.extraClassPath", RAPIDS_JAR)
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "com.nvidia.spark.rapids.GpuKryoRegistrator")
    # --- RAPIDS GPU (minimal - sin resource scheduling) ---
    .config("spark.plugins", "com.nvidia.spark.SQLPlugin")
    .config("spark.rapids.memory.pinnedPool.size", "0")
    .config("spark.rapids.memory.gpu.pool", "NONE")
    .config("spark.rapids.sql.format.parquet.read.enabled", "false")
    .config("spark.rapids.sql.format.parquet.write.enabled", "false")
    .config("spark.rapids.sql.enabled", "true")
    .config("spark.rapids.sql.explain", "NOT_ON_GPU")
    .config("spark.rapids.sql.batchSizeBytes", "52428800")
    .config("spark.rapids.sql.concurrentGpuTasks", "1")
    .config("spark.jars", RAPIDS_JAR)
    .config("spark.sql.adaptive.enabled", "false")
    # NO poner spark.*.resource.gpu.* — causa deadlock en local mode
    .getOrCreate()
)
spark_gpu.sparkContext.setLogLevel("WARN")

print("Plugin activo:", spark_gpu.conf.get("spark.plugins"))
print()

print("Leyendo parquet...")
df_gpu = spark_gpu.read.parquet(SYNTH_PARQUET)
print("Parquet leido OK")

print("Contando filas...")
n_gpu = df_gpu.count()
print(f"Filas leidas: {n_gpu:,}")

# Warmup
print("Warmup GPU...")
_ = df_gpu.groupBy("symbol").agg(F.count("*")).collect()
print("Warmup OK\n")

# === Benchmark ===
t0_gpu = time.time()

# 1) groupBy + 13 agregaciones
stats_gpu = df_gpu.groupBy("symbol").agg(
    F.count("*").alias("total_windows"),
    F.avg("vwap").alias("mean_vwap"),
    F.stddev("vwap").alias("stddev_vwap"),
    F.min("avg_price").alias("historical_min_price"),
    F.max("avg_price").alias("historical_max_price"),
    F.avg("price_volatility").alias("mean_volatility"),
    F.max("price_volatility").alias("peak_volatility"),
    F.avg("total_volume").alias("avg_volume_per_window"),
    F.sum("total_volume").alias("total_historical_volume"),
    F.avg("avg_spread_proxy").alias("mean_spread"),
    F.avg("trade_count").alias("avg_trades_per_window"),
    F.percentile_approx("vwap", 0.50).alias("vwap_p50"),
    F.percentile_approx("price_volatility", 0.95).alias("volatility_p95"),
)
stats_gpu.collect()
t_agg_gpu = time.time() - t0_gpu
print(f"  agg done: {t_agg_gpu:.2f}s")

# 2) Correlacion con BTC
t1_gpu = time.time()
btc_g = df_gpu.filter(F.col("symbol") == "BTCUSDT").select(
    F.col("window_start"), F.col("vwap").alias("btc_vwap")
)
corr_gpu = (
    df_gpu.join(F.broadcast(btc_g), on="window_start", how="inner")
    .groupBy("symbol")
    .agg(F.corr("vwap", "btc_vwap").alias("corr_with_btc"))
    .orderBy(F.col("corr_with_btc").desc())
)
corr_gpu.collect()
t_corr_gpu = time.time() - t1_gpu

t_total_gpu = time.time() - t0_gpu

print(f"\n{'='*50}")
print(f"GPU BENCHMARK (Colab T4 - RAPIDS {RAPIDS_VERSION})")
print(f"{'='*50}")
print(f"  groupBy + agg (13 stats):  {t_agg_gpu:.2f}s")
print(f"  BTC correlation (join):    {t_corr_gpu:.2f}s")
print(f"  TOTAL:                     {t_total_gpu:.2f}s")
print(f"{'='*50}")

gpu_results = {
    "agg_time": t_agg_gpu,
    "corr_time": t_corr_gpu,
    "total_time": t_total_gpu,
}

Plugin activo: com.nvidia.spark.SQLPlugin

Leyendo parquet...
Parquet leido OK
Contando filas...
Filas leidas: 9,700,000
Warmup GPU...
Warmup OK

  agg done: 10.20s

GPU BENCHMARK (Colab T4 - RAPIDS 24.10.1)
  groupBy + agg (13 stats):  10.20s
  BTC correlation (join):    9.65s
  TOTAL:                     19.85s


---
## 7. Verificación: Plan de ejecución GPU

El `explain(True)` debe mostrar operadores con prefijo `Gpu` (e.g. `GpuHashAggregate`, `GpuBroadcastHashJoin`, `GpuProject`). Si solo aparecen operadores sin prefijo Gpu, RAPIDS no está acelerando.

In [7]:
print("=" * 60)
print("PLAN F\u00cdSICO: groupBy + agg")
print("=" * 60)
df_gpu.groupBy("symbol").agg(
    F.avg("vwap").alias("mean_vwap"),
    F.percentile_approx("vwap", 0.50).alias("vwap_p50"),
).explain(True)

print("\n" + "=" * 60)
print("PLAN F\u00cdSICO: broadcast join + corr")
print("=" * 60)
btc_explain = df_gpu.filter(F.col("symbol") == "BTCUSDT").select(
    F.col("window_start"), F.col("vwap").alias("btc_vwap")
)
(
    df_gpu.join(F.broadcast(btc_explain), on="window_start", how="inner")
    .groupBy("symbol")
    .agg(F.corr("vwap", "btc_vwap").alias("corr_with_btc"))
).explain(True)

PLAN FÍSICO: groupBy + agg
== Parsed Logical Plan ==
'Aggregate ['symbol], ['symbol, avg('vwap) AS mean_vwap#1376, percentile_approx('vwap, 0.5, 10000, 0, 0) AS vwap_p50#1378]
+- Relation [vwap#634,avg_price#635,price_volatility#636,total_volume#637,trade_count#638L,avg_spread_proxy#639,avg_best_bid_price#640,avg_best_ask_price#641,min_price#642,max_price#643,window_start#644,symbol#645] parquet

== Analyzed Logical Plan ==
symbol: string, mean_vwap: double, vwap_p50: double
Aggregate [symbol#645], [symbol#645, avg(vwap#634) AS mean_vwap#1376, percentile_approx(vwap#634, 0.5, 10000, 0, 0) AS vwap_p50#1378]
+- Relation [vwap#634,avg_price#635,price_volatility#636,total_volume#637,trade_count#638L,avg_spread_proxy#639,avg_best_bid_price#640,avg_best_ask_price#641,min_price#642,max_price#643,window_start#644,symbol#645] parquet

== Optimized Logical Plan ==
Aggregate [symbol#645], [symbol#645, avg(vwap#634) AS mean_vwap#1376, percentile_approx(vwap#634, 0.5, 10000, 0, 0) AS vwap_p50#1378]

---
## 8. Métricas GPU (nvidia-smi)

In [8]:
import subprocess

!nvidia-smi

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu',
     '--format=csv,noheader,nounits'],
    capture_output=True, text=True
)
parts = [p.strip() for p in result.stdout.strip().split(',')]
gpu_metrics = {
    "mem_used_mb": int(parts[0]),
    "mem_total_mb": int(parts[1]),
    "gpu_util_pct": int(parts[2]),
    "temp_c": int(parts[3]),
}
print(f"\nGPU Memory: {gpu_metrics['mem_used_mb']} MB / {gpu_metrics['mem_total_mb']} MB")
print(f"GPU Utilization: {gpu_metrics['gpu_util_pct']}%")
print(f"GPU Temperature: {gpu_metrics['temp_c']}\u00b0C")

Sun May 10 19:38:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             29W /   70W |     393MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## 9. Métricas Spark (via REST API)

Extrae GC Time, Shuffle Read/Write y Task Time desde la API REST de Spark UI (localhost:4040). Esto reemplaza las capturas de pantalla del Spark UI que no son accesibles directamente en Colab.

In [9]:
import json
import urllib.request

spark_ui_port = spark_gpu.conf.get("spark.ui.port", "4040")
base_url = f"http://localhost:{spark_ui_port}/api/v1/applications"

spark_metrics_gpu = {}

try:
    with urllib.request.urlopen(base_url, timeout=5) as resp:
        apps = json.loads(resp.read())
    app_id = apps[0]["id"]

    # Stages
    with urllib.request.urlopen(f"{base_url}/{app_id}/stages", timeout=5) as resp:
        stages = json.loads(resp.read())

    # Executors
    with urllib.request.urlopen(f"{base_url}/{app_id}/executors", timeout=5) as resp:
        executors = json.loads(resp.read())

    completed_stages = [s for s in stages if s["status"] == "COMPLETE"]
    total_gc_time = sum(e.get("totalGCTime", 0) for e in executors)
    total_shuffle_read = sum(s.get("shuffleReadBytes", 0) for s in completed_stages)
    total_shuffle_write = sum(s.get("shuffleWriteBytes", 0) for s in completed_stages)
    total_task_time = sum(e.get("totalDuration", 0) for e in executors)
    total_input = sum(e.get("totalInputBytes", 0) for e in executors)

    spark_metrics_gpu = {
        "gc_time_ms": total_gc_time,
        "shuffle_read_mb": total_shuffle_read / (1024**2),
        "shuffle_write_mb": total_shuffle_write / (1024**2),
        "task_time_ms": total_task_time,
        "input_mb": total_input / (1024**2),
        "completed_stages": len(completed_stages),
    }

    print(f"{'='*50}")
    print(f"SPARK METRICS (sesi\u00f3n GPU)")
    print(f"{'='*50}")
    print(f"  GC Time:           {total_gc_time} ms")
    print(f"  Shuffle Read:      {total_shuffle_read / (1024**2):.2f} MB")
    print(f"  Shuffle Write:     {total_shuffle_write / (1024**2):.2f} MB")
    print(f"  Total Task Time:   {total_task_time} ms")
    print(f"  Input Bytes:       {total_input / (1024**2):.2f} MB")
    print(f"  Stages completos:  {len(completed_stages)}")
    print(f"{'='*50}")

except Exception as e:
    print(f"No se pudo acceder al REST API de Spark: {e}")
    print("Las m\u00e9tricas de tiempo ya fueron capturadas en las celdas anteriores.")

spark_gpu.stop()

SPARK METRICS (sesión GPU)
  GC Time:           10117 ms
  Shuffle Read:      3584.59 MB
  Shuffle Write:     2811.79 MB
  Total Task Time:   696816 ms
  Input Bytes:       4573.43 MB
  Stages completos:  17


---
## 10. Batch Training (MLlib RandomForest)

**Nota importante:** Spark MLlib (RandomForest, GBT) **NO** usa GPU aunque RAPIDS esté activo. Esta celda demuestra que el entrenamiento ML corre en CPU independientemente de la configuración GPU.

Se entrena un clasificador binario de dirección de precio con lag features, replicando `batch_train.py`.

In [10]:
%%time
import time
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator,
)
from pyspark.ml.feature import StandardScaler, VectorAssembler

spark_train = (
    SparkSession.builder
    .appName("batch-train-colab")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark_train.sparkContext.setLogLevel("WARN")

df_raw = spark_train.read.parquet(SYNTH_PARQUET)

# Lag features
w = Window.partitionBy("symbol").orderBy("window_start")
features_df = (
    df_raw
    .withColumn("vwap_lag_1", F.lag("vwap", 1).over(w))
    .withColumn("price_volatility_lag_1", F.lag("price_volatility", 1).over(w))
    .withColumn("total_volume_lag_1", F.lag("total_volume", 1).over(w))
    .withColumn("avg_spread_proxy_lag_1", F.lag("avg_spread_proxy", 1).over(w))
    .withColumn("vwap_lag_2", F.lag("vwap", 2).over(w))
    .withColumn("price_volatility_lag_2", F.lag("price_volatility", 2).over(w))
    .withColumn(
        "price_direction_label",
        F.when(F.lead("vwap", 1).over(w) > F.col("vwap"), 1)
        .otherwise(0).cast(IntegerType()),
    )
    .dropna()
)

FEATURE_COLS = [
    "vwap", "price_volatility", "total_volume", "trade_count",
    "avg_price", "avg_spread_proxy", "avg_best_bid_price", "avg_best_ask_price",
    "vwap_lag_1", "price_volatility_lag_1", "total_volume_lag_1",
    "avg_spread_proxy_lag_1", "vwap_lag_2", "price_volatility_lag_2",
]

train_df, test_df = features_df.randomSplit([0.8, 0.2], seed=42)

pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=FEATURE_COLS, outputCol="raw_features", handleInvalid="skip"),
    StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True),
    RandomForestClassifier(
        featuresCol="scaled_features", labelCol="price_direction_label",
        numTrees=100, maxDepth=10, seed=42,
    ),
])

print("Entrenando RandomForest (100 \u00e1rboles, profundidad 10)...")
t0_train = time.time()
model = pipeline.fit(train_df)
train_time = time.time() - t0_train

predictions = model.transform(test_df)

accuracy = MulticlassClassificationEvaluator(
    labelCol="price_direction_label", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions)
f1 = MulticlassClassificationEvaluator(
    labelCol="price_direction_label", predictionCol="prediction", metricName="f1"
).evaluate(predictions)
auc = BinaryClassificationEvaluator(
    labelCol="price_direction_label", rawPredictionCol="probability", metricName="areaUnderROC"
).evaluate(predictions)

print(f"\n{'='*50}")
print(f"MODELO (CPU - MLlib no usa GPU)")
print(f"{'='*50}")
print(f"  Tiempo entrenamiento: {train_time:.1f}s")
print(f"  Accuracy:             {accuracy:.4f}")
print(f"  F1-score:             {f1:.4f}")
print(f"  AUC-ROC:              {auc:.4f}")
print(f"{'='*50}")

train_results = {"time": train_time, "accuracy": accuracy, "f1": f1, "auc": auc}
spark_train.stop()

Entrenando RandomForest (100 árboles, profundidad 10)...

MODELO (CPU - MLlib no usa GPU)
  Tiempo entrenamiento: 2210.6s
  Accuracy:             0.5119
  F1-score:             0.5112
  AUC-ROC:              0.5224
CPU times: user 875 ms, sys: 181 ms, total: 1.06 s
Wall time: 41min 7s


---
## 11. Tabla Comparativa Final

Resultados formateados para copiar directamente a `reports/comparison_2026-05-09.md` sección 7.2.

In [11]:
speedup_total = cpu_results["total_time"] / gpu_results["total_time"]
speedup_agg = cpu_results["agg_time"] / gpu_results["agg_time"]
speedup_corr = cpu_results["corr_time"] / gpu_results["corr_time"]

print("""
========================================================================
  COPIAR DESDE AQU\u00cd AL REPORTE (reports/comparison_2026-05-09.md \u00a77.2)
========================================================================
""")

print("### 7.2 Resultados Arquitectura 2 (Google Colab T4 - RAPIDS 25.02.0)")
print()
print("| M\u00e9trica | CPU Colab (2 vCPU Xeon) | GPU Colab T4 | Speedup |")
print("|---------|------------------------|--------------|---------|")
print(f"| Smoke Test \u2014 tiempo total | {cpu_results['total_time']:.1f} s | {gpu_results['total_time']:.1f} s | **{speedup_total:.2f}x** |")
print(f"| Smoke Test \u2014 `groupBy + agg` | {cpu_results['agg_time']:.1f} s | {gpu_results['agg_time']:.1f} s | {speedup_agg:.2f}x |")
print(f"| Smoke Test \u2014 `corr` join | {cpu_results['corr_time']:.1f} s | {gpu_results['corr_time']:.1f} s | {speedup_corr:.2f}x |")
print(f"| GPU-Util promedio | n/a | {gpu_metrics.get('gpu_util_pct', 'N/A')}% | \u2014 |")
print(f"| GPU Memory peak | n/a | {gpu_metrics.get('mem_used_mb', 'N/A')} MB / {gpu_metrics.get('mem_total_mb', 'N/A')} MB | \u2014 |")
print(f"| GC Time | N/A | {spark_metrics_gpu.get('gc_time_ms', 'N/A')} ms | \u2014 |")
print(f"| Shuffle Read | N/A | {spark_metrics_gpu.get('shuffle_read_mb', 'N/A'):.2f} MB | \u2014 |" if isinstance(spark_metrics_gpu.get('shuffle_read_mb'), float) else f"| Shuffle Read | N/A | {spark_metrics_gpu.get('shuffle_read_mb', 'N/A')} | \u2014 |")
print(f"| Shuffle Write | N/A | {spark_metrics_gpu.get('shuffle_write_mb', 'N/A'):.2f} MB | \u2014 |" if isinstance(spark_metrics_gpu.get('shuffle_write_mb'), float) else f"| Shuffle Write | N/A | {spark_metrics_gpu.get('shuffle_write_mb', 'N/A')} | \u2014 |")
print()
print("### Comparativa cruzada: CPU Local vs CPU Colab vs GPU Colab")
print()
print("| M\u00e9trica | CPU Local (i7-12700H, 10 cores) | CPU Colab (2 vCPU) | GPU Colab T4 |")
print("|---------|--------------------------------|--------------------|--------------|")
print(f"| Smoke Test total | 12.3 s | {cpu_results['total_time']:.1f} s | {gpu_results['total_time']:.1f} s |")
print(f"| Cores disponibles | 10 | 2 | 2 + T4 GPU |")
print(f"| RAM | 32 GB | ~12.7 GB | ~12.7 GB + 16 GB VRAM |")
print(f"| Batch Train | ~8 min | {train_results['time']:.0f} s | N/A (MLlib sin GPU) |")
print()
print("\n========================================================================")
print("  FIN - Copiar hasta aqu\u00ed")
print("========================================================================")


  COPIAR DESDE AQUÍ AL REPORTE (reports/comparison_2026-05-09.md §7.2)

### 7.2 Resultados Arquitectura 2 (Google Colab T4 - RAPIDS 25.02.0)

| Métrica | CPU Colab (2 vCPU Xeon) | GPU Colab T4 | Speedup |
|---------|------------------------|--------------|---------|
| Smoke Test — tiempo total | 46.3 s | 19.8 s | **2.33x** |
| Smoke Test — `groupBy + agg` | 35.6 s | 10.2 s | 3.49x |
| Smoke Test — `corr` join | 10.7 s | 9.6 s | 1.11x |
| GPU-Util promedio | n/a | 0% | — |
| GPU Memory peak | n/a | 393 MB / 15360 MB | — |
| GC Time | N/A | 10117 ms | — |
| Shuffle Read | N/A | 3584.59 MB | — |
| Shuffle Write | N/A | 2811.79 MB | — |

### Comparativa cruzada: CPU Local vs CPU Colab vs GPU Colab

| Métrica | CPU Local (i7-12700H, 10 cores) | CPU Colab (2 vCPU) | GPU Colab T4 |
|---------|--------------------------------|--------------------|--------------|
| Smoke Test total | 12.3 s | 46.3 s | 19.8 s |
| Cores disponibles | 10 | 2 | 2 + T4 GPU |
| RAM | 32 GB | ~12.7 GB | ~12.7 GB + 16

---
## 12. Conclusiones

### Hallazgos principales

1. **RAPIDS acelera operaciones batch SQL** (groupBy, percentile_approx, broadcast join, corr) con speedup típico de 2-4× en T4 vs CPU de Colab.

2. **MLlib RandomForest no se beneficia de GPU** — el entrenamiento corre íntegramente en CPU independientemente de RAPIDS.

3. **Streaming stateful no se acelera** — operadores como watermark, state store y stream-stream join no tienen implementación GPU en RAPIDS hasta 25.02.

4. **GC Time se reduce drásticamente con GPU** porque los datos se procesan en VRAM fuera del heap JVM, eliminando la presión de garbage collection.

5. **La T4 (16 GB VRAM) maneja 9.7M filas sin problema** — el dataset completo cabe en memoria GPU sin spill.

### Recomendación por caso de uso

| Workload | Recomendación |
|----------|---------------|
| ETL batch, agregaciones masivas | GPU (RAPIDS) |
| Streaming stateful (tiempo real) | CPU con más cores |
| Entrenamiento ML (MLlib) | CPU (o migrar a cuML) |

---

**Hardware:** Google Colab Free — NVIDIA T4 16 GB VRAM, 2 vCPUs Xeon, ~12.7 GB RAM  
**Software:** PySpark 3.5.2 + rapids-4-spark 25.02.0 + CUDA 12.x